[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/work_energy.ipynb)

# Work, energy and power full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one
step), **tier 2** core problems (work by several forces, the work–energy theorem, areas under a force graph,
conservation of energy, loops, springs, power and collisions) and **tier 3** stretch problems (combine ideas, spot the
trap, work backwards). Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same answers in a few lines of Python, so
you can change the numbers and run them again. And the course's simulations, with sliders you move yourself: two carts
meeting through a springy bumper (change the masses), and the roller coaster's loop from the course's opening question
(change the launch speed and the loop's radius, and watch the energy bars).

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Two decimals are enough, in the unit the question names
(the box's label shows it); you can type the unit too, `420 J`, or the arithmetic, `0.5*70*6**2` or `sqrt(2*9.8*20)`.
Numbers without thousands separators: `120000`, not `120,000`. For several answers, separate them with commas, in the
order the question lists them. Press **show working** to see the steps. You can also put your answer between the
quotes in a cell's last line, `practice("P1.1", "")`, and run the cell: it checks straight away. g = 9.8 m/s²
throughout, unless a question says otherwise.

**The cheat sheet.** Every formula from the course on two pages:
[work, energy and power cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/work_energy_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the checker and the simulator: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # bodies in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# the course's colours: velocity SKY · acceleration MINT · force CORAL · energy VIOLET · bodies CHALK · the one thing to
# look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

g = 9.8                                               # m/s²

def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """The course's simulator: fixed-step fourth-order Runge–Kutta. deriv(t, s) gives ds/dt; it returns the list of
    (t, state) samples, ending at t_end or at the first step where stop(t, s) is True."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t += dt
        out.append((t, s))
        if stop is not None and stop(t, s): break
    return out

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, sqrt(…) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id == "pi": return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = r"\s*(?:[a-zA-Zµ°%²/·]+\s*)+$"                  # a unit typed after the number: J, kJ, m/s, N/m, MW, °, …

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    for t in (s, re.sub(UNIT, "", s)):
        try: return calc(t)
        except Exception: pass
    return None

def right(got, want, tol):
    return got is not None and abs(got - want) <= tol + 1e-9   # + 1e-9: an answer on the half (0.375 → 0.38) passes

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own questions, answers and working ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Work',
     "q": 'A trolley is pushed 12 m along a corridor by a level push of 35 N. How much work does the push do, in joules?',
     "parts": ["the push's work (J)"],
     "answer": [420.0], "tol": [0.005],
     "working": 'W = F d = 35 × 12 = 420 J (the push is along the motion, cos 0° = 1)'},
    {"id": 'P1.2', "tier": 1, "group": 'Work',
     "q": 'A rope pulls a sledge with 80 N at 40° above the ground for 10 m. How much work does it do, in joules?',
     "parts": ["the rope's work (J)"],
     "answer": [612.8355544951825], "tol": [0.005],
     "working": 'W = F d cos θ = 80 × 10 × cos 40° = 800 × 0.7660 = 612.84 J'},
    {"id": 'P1.3', "tier": 1, "group": 'Kinetic energy and the work–energy theorem',
     "q": 'What is the kinetic energy of a 70 kg runner at 6 m/s, in joules?',
     "parts": ['kinetic energy (J)'],
     "answer": [1260.0], "tol": [0.005],
     "working": 'K = ½ m v² = ½ × 70 × 36 = 1,260 J'},
    {"id": 'P1.4', "tier": 1, "group": 'Kinetic energy and the work–energy theorem',
     "q": 'A 0.05 kg ball has 10 J of kinetic energy. How fast is it going, in m/s?',
     "parts": ['speed (m/s)'],
     "answer": [20.0], "tol": [0.005],
     "working": 'v = √(2K/m) = √(20/0.05) = √400 = 20.00 m/s'},
    {"id": 'P1.5', "tier": 1, "group": 'Work by a variable force',
     "q": 'A force rises steadily from 0 to 50 N while it pushes a box 4 m. How much work does it do, in joules?',
     "parts": ['work (J)'],
     "answer": [100.0], "tol": [0.005],
     "working": 'The F–x graph is a triangle: area = ½ × 4 × 50 = 100 J'},
    {"id": 'P1.6', "tier": 1, "group": 'Potential energy and conservation',
     "q": 'How much potential energy does a 5 kg bag gain when lifted onto a 2 m high shelf, in joules?',
     "parts": ['potential energy gained (J)'],
     "answer": [98.0], "tol": [0.005],
     "working": 'U = m g h = 5 × 9.8 × 2 = 98 J'},
    {"id": 'P1.7', "tier": 1, "group": 'Potential energy and conservation',
     "q": 'A stone falls 20 m from rest. Ignoring air, how fast is it going at the bottom, in m/s?',
     "parts": ['speed at the bottom (m/s)'],
     "answer": [19.79898987322333], "tol": [0.005],
     "working": '½ m v² = m g h → v = √(2 g h) = √392 = 19.80 m/s'},
    {"id": 'P1.8', "tier": 1, "group": 'Motion in a vertical circle',
     "q": 'A ball on a string is swung in a vertical circle of radius 0.8 m. What is the least speed it can have at the top with the string still tight, in m/s?',
     "parts": ['least speed at the top (m/s)'],
     "answer": [2.8000000000000003], "tol": [0.005],
     "working": 'At the top, v²/R ≥ g → v ≥ √(g R) = √7.84 = 2.80 m/s'},
    {"id": 'P1.9', "tier": 1, "group": 'The energy in a spring',
     "q": 'A spring with k = 300 N/m is stretched 0.2 m. How much energy does it store, in joules?',
     "parts": ['energy stored (J)'],
     "answer": [6.0], "tol": [0.005],
     "working": 'U = ½ k x² = ½ × 300 × 0.04 = 6 J'},
    {"id": 'P1.10', "tier": 1, "group": 'Power',
     "q": 'A motor lifts a 50 kg load 12 m in 20 s at steady speed. What is its average power, in watts?',
     "parts": ['average power (W)'],
     "answer": [294.00000000000006], "tol": [0.005],
     "working": 'P = W/t = m g h / t = 5,880 / 20 = 294 W'},
    {"id": 'P2.1', "tier": 2, "group": 'Work',
     "q": 'A 30 kg box is dragged 8 m across a floor by a level pull of 100 N against 70 N of friction. Give the work done by the pull, by friction and by gravity, and the net work, in joules.',
     "parts": ['work by the pull (J)', 'work by friction (J)', 'work by gravity (J)', 'net work (J)'],
     "answer": [800.0, -560.0, 0.0, 240.0], "tol": [0.005, 0.005, 0.005, 0.005],
     "working": 'Pull +800 J; friction −560 J (it acts against the motion: cos 180° = −1); gravity 0 (at 90° to the motion); net +240 J'},
    {"id": 'P2.2', "tier": 2, "group": 'Kinetic energy and the work–energy theorem',
     "q": 'A 1,200 kg car speeds up from 10 m/s to 25 m/s. How much net work was done on it, in kilojoules?',
     "parts": ['net work (kJ)'],
     "answer": [315.0], "tol": [0.005],
     "working": 'W = ½ m (v² − u²) = ½ × 1,200 × (625 − 100) = 315,000 J = 315 kJ'},
    {"id": 'P2.3', "tier": 2, "group": 'Kinetic energy and the work–energy theorem',
     "q": 'A cyclist and bike, 80 kg in all, brake from 9 m/s to rest in 12 m. What is the average braking force, in newtons?',
     "parts": ['average braking force (N)'],
     "answer": [270.0], "tol": [0.005],
     "working": 'Work–energy theorem: F d = ½ m v² → F = ½ × 80 × 81 / 12 = 270 N'},
    {"id": 'P2.4', "tier": 2, "group": 'Work by a variable force',
     "q": 'A force on a 4 kg cart rises from 0 to 30 N over the first 2 m, then stays at 30 N for 3 m. From rest, how fast is the cart going at the end, in m/s?',
     "parts": ['speed at the end (m/s)'],
     "answer": [7.745966692414834], "tol": [0.005],
     "working": 'Area = ½ × 2 × 30 + 3 × 30 = 120 J; v = √(2 × 120/4) = √60 = 7.75 m/s'},
    {"id": 'P2.5', "tier": 2, "group": 'Potential energy and conservation',
     "q": "A skateboarder starts from rest at the top of a 3.2 m ramp. Ignoring friction, how fast is she going at the bottom, in m/s? Does the ramp's shape matter?",
     "parts": ['speed at the bottom (m/s)'],
     "answer": [7.919595949289333], "tol": [0.005],
     "working": "v = √(2 g h) = √62.72 = 7.92 m/s; no: gravity's work depends only on the height"},
    {"id": 'P2.6', "tier": 2, "group": 'Potential energy and conservation',
     "q": 'A 0.5 kg ball is thrown straight up at 12 m/s. Ignoring air, how high does it rise, in metres?',
     "parts": ['height (m)'],
     "answer": [7.346938775510203], "tol": [0.005],
     "working": '½ m u² = m g h → h = u²/(2g) = 144/19.6 = 7.35 m'},
    {"id": 'P2.7', "tier": 2, "group": 'Motion in a vertical circle',
     "q": 'A roller-coaster loop on a ramp has radius 6 m. With no friction, what is the least speed at the bottom for a ball rolling freely inside it to stay on the track at the top, in m/s?',
     "parts": ['least speed at the bottom (m/s)'],
     "answer": [17.146428199482248], "tol": [0.005],
     "working": 'v ≥ √(5 g R) = √294 = 17.15 m/s'},
    {"id": 'P2.8', "tier": 2, "group": 'The energy in a spring',
     "q": 'A 0.2 kg toy is fired by a spring (k = 500 N/m) squeezed 4 cm. How fast does it leave, in m/s?',
     "parts": ['launch speed (m/s)'],
     "answer": [2.0], "tol": [0.005],
     "working": '½ k x² = 0.4 J = ½ m v² → v = √(2 × 0.4 / 0.2) = √4 = 2.00 m/s'},
    {"id": 'P2.9', "tier": 2, "group": 'Power',
     "q": "A car's engine delivers 45 kW while the car holds 30 m/s on a level road. What resisting force is it working against, in newtons?",
     "parts": ['resisting force (N)'],
     "answer": [1500.0], "tol": [0.005],
     "working": 'P = F v → F = 45,000 / 30 = 1,500 N'},
    {"id": 'P2.10', "tier": 2, "group": 'Collisions',
     "q": 'A 3 kg cart at 4 m/s hits a 1 kg cart at rest and they stick. What is their speed after, in m/s, and how much kinetic energy is lost, in joules?',
     "parts": ['speed after (m/s)', 'kinetic energy lost (J)'],
     "answer": [3.0, 6.0], "tol": [0.005, 0.005],
     "working": 'Momentum: 3 × 4 = 4 v → v = 3 m/s; kinetic energy 24 J → 18 J, so 6 J lost'},
    {"id": 'P3.1', "tier": 3, "group": 'Work',
     "q": 'A suitcase is pulled 20 m with 50 N. The work done is 707.11 J. At what angle to the ground is the handle, in degrees?',
     "parts": ["the handle's angle (°)"],
     "answer": [44.99973918388208], "tol": [0.005],
     "working": 'cos θ = W/(F d) = 707.11/1,000 = 0.7071 → θ = 45.00°'},
    {"id": 'P3.2', "tier": 3, "group": 'Kinetic energy and the work–energy theorem',
     "q": 'Car A (1,000 kg) goes at 20 m/s; car B (2,000 kg) at 10 m/s. Which has more kinetic energy, and how many times more? With the same braking force, how do their stopping distances compare?',
     "parts": ["how many times more: A's ÷ B's"],
     "answer": [2.0], "tol": [0.005],
     "working": 'A: 200 kJ; B: 100 kJ. A has twice as much, so with the same force it needs twice the distance (F d = K). Speed counts squared, mass only once'},
    {"id": 'P3.3', "tier": 3, "group": 'Work by a variable force',
     "q": 'A 2 kg cart reaches 6 m/s, pushed from rest by a force that falls steadily to zero over 3 m. What was the force at the start, in newtons?',
     "parts": ['the force at the start (N)'],
     "answer": [24.0], "tol": [0.005],
     "working": 'Work = ½ m v² = 36 J = triangle area ½ × F0 × 3 → F0 = 24 N'},
    {"id": 'P3.4', "tier": 3, "group": 'Potential energy and conservation',
     "q": 'A 40 kg child slides down a 3 m high slide and reaches 6 m/s. The slide is 6 m long. How much energy went to heat, and what was the average friction force, in joules and newtons?',
     "parts": ['energy to heat (J)', 'average friction force (N)'],
     "answer": [456.0, 76.0], "tol": [0.005, 0.005],
     "working": 'm g h = 1,176 J; ½ m v² = 720 J; heat 456 J; friction = 456/6 = 76 N'},
    {"id": 'P3.5', "tier": 3, "group": 'Motion in a vertical circle',
     "q": "A toy car runs down a smooth ramp into a loop of radius 0.15 m. From what least height above the loop's bottom must it start to stay on the track at the top, in metres?",
     "parts": ['least starting height (m)'],
     "answer": [0.375], "tol": [0.005],
     "working": 'Needs v² ≥ 5 g R at the bottom; m g h = ½ m v² → h = 2.5 R = 0.375 m. (Mass and g both cancel.)'},
    {"id": 'P3.6', "tier": 3, "group": 'Motion in a vertical circle',
     "q": "A stunt rider enters a loop of radius 5 m at 16 m/s. With no friction, what is the speed at the top, in m/s, and by what fraction of the rider's weight does the seat press at the top?",
     "parts": ['speed at the top (m/s)', "the seat's push, as a share of the weight"],
     "answer": [7.745966692414834, 0.22448979591836737], "tol": [0.005, 0.005],
     "working": 'v_top² = 256 − 196 = 60 → 7.75 m/s; v²/(gR) − 1 = 60/49 − 1 = 0.22'},
    {"id": 'P3.7', "tier": 3, "group": 'The energy in a spring',
     "q": 'A buffer spring must stop a 500 kg cart at 3 m/s within 0.3 m. What spring constant does it need, in N/m? What if the cart came at 6 m/s?',
     "parts": ['k at 3 m/s (N/m)', 'k at 6 m/s (N/m)'],
     "answer": [50000.0, 200000.0], "tol": [0.005, 0.005],
     "working": '½ k x² = ½ m v² → k = m v²/x² = 500 × 9/0.09 = 50,000 N/m; at twice the speed, four times: 200,000 N/m'},
    {"id": 'P3.8', "tier": 3, "group": 'Collisions',
     "q": "A 1 kg cart at 5 m/s hits a 4 kg cart at rest. The collision is elastic. Find both velocities after, in m/s (positive = the first cart's direction), and check momentum and kinetic energy.",
     "parts": ["first cart's velocity after (m/s)", "second cart's velocity after (m/s)"],
     "answer": [-3.0, 2.0], "tol": [0.005, 0.005],
     "working": 'v1 = (1 − 4)/5 × 5 = −3 m/s (it bounces back); v2 = 2 × 1/5 × 5 = 2 m/s; momentum 5 = −3 + 8 ✓; energy 12.5 = 4.5 + 8 ✓'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes (with or without its unit), a wrong one doesn't
assert right(read_part("35*12"), 420, 0.005) and right(read_part("420 J"), 420, 0.005) and not right(read_part("421"), 420, 0.005)
assert right(read_part("−560"), -560, 0.005) and right(read_part("sqrt(2*9.8*20) m/s"), 19.79899, 0.005)
assert right(read_part("0.38"), 0.375, 0.005) and read_part("ten") is None

## Tier 1 · Warm-up

One idea, one step: the work of a level push and of a slanted rope, kinetic energy both ways, the area under a force
that rises, potential energy, a speed from a drop, the slowest speed at the top of a circle, a stretched spring's
energy, and a motor's power.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each warm-up is one line of arithmetic. Python's `math.cos` takes an angle in **radians**, so turn degrees into
radians first with `math.radians`. Each line below computes one
warm-up answer and checks it against the practice set.

In [ ]:
W_push = 35 * 12                                                  # P1.1: W = F d, the push along the motion
W_rope = 80 * 10 * math.cos(math.radians(40))                     # P1.2: W = F d cos θ, θ in radians
K_run = 0.5 * 70 * 6 ** 2                                         # P1.3: K = ½ m v²
v_ball = math.sqrt(2 * 10 / 0.05)                                 # P1.4: v = √(2K/m)
W_tri = 0.5 * 50 * 4                                              # P1.5: the triangle under F–x
U_bag = 5 * g * 2                                                 # P1.6: U = m g h
v_stone = math.sqrt(2 * g * 20)                                   # P1.7: m g h = ½ m v²
v_top = math.sqrt(g * 0.8)                                        # P1.8: v²/R ≥ g at the top
U_spring = 0.5 * 300 * 0.2 ** 2                                   # P1.9: U = ½ k x²
P_motor = 50 * g * 12 / 20                                        # P1.10: P = W/t

print(f"push {W_push} J · rope {W_rope:.2f} J · runner {K_run} J · ball {v_ball:.2f} m/s · triangle {W_tri} J · "
      f"bag {U_bag:.1f} J")
print(f"stone {v_stone:.2f} m/s · top of the circle {v_top:.2f} m/s · spring {U_spring:.2f} J · motor {P_motor:.0f} W")
for k, v in (("P1.1", [W_push]), ("P1.2", [W_rope]), ("P1.3", [K_run]), ("P1.4", [v_ball]), ("P1.5", [W_tri]),
             ("P1.6", [U_bag]), ("P1.7", [v_stone]), ("P1.8", [v_top]), ("P1.9", [U_spring]), ("P1.10", [P_motor])):
    assert agrees(k, v), k

### A push that falls: the work is the area

The course's trolley: 5 kg, pushed from rest by a force that falls steadily from 30 N to 10 N over 4 m. The work is
the area under the force–distance line, 80 J, so the trolley ends at √(2 × 80 / 5) ≈ 5.66 m/s. The simulator pushes
the trolley with that changing force, step by step, and gets the same speed.

In [ ]:
m_t, F0, F1, L = 5.0, 30.0, 10.0, 4.0
F = lambda x: F0 + (F1 - F0) * x / L
xs = np.linspace(0, L, 9)                                          # 0.5 m strips
area = float(np.sum((xs[1:] - xs[:-1]) * (F(xs[1:]) + F(xs[:-1])) / 2))
run = simulate(lambda t, s: (s[1], F(s[0]) / m_t), (0.0, 0.0), 10, 1e-4, stop=lambda t, s: s[0] >= L)
t_end, (x_end, v_end) = run[-1]
print(f"area under F–x: {area:.2f} J → speed √(2W/m) = {math.sqrt(2 * area / m_t):.4f} m/s; "
      f"simulated: {v_end:.4f} m/s at x = {x_end:.4f} m after {t_end:.3f} s")
assert abs(area - 80) < 1e-9 and abs(v_end - math.sqrt(32)) < 2e-3                          # the last step overshoots 4 m a hair

fig, ax = plt.subplots(figsize=(11.5, 4.2))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.16)
ax.fill_between([0, L], [F0, F1], color=VIOLET, alpha=0.35, lw=0)
ax.plot([0, L], [F0, F1], color=CORAL)
ax.set_xlim(0, L); ax.set_ylim(0, 35); ax.set_xlabel("distance x (m)"); ax.set_ylabel("push F (N)")
fig.suptitle(f"The area under the push: {area:.0f} J of work → {v_end:.2f} m/s", x=0.08, ha="left", fontsize=15,
             color=GOLD)
footnote(fig); plt.show()

## Tier 2 · Core

The course's standard problems, several steps each: the work of every force on a dragged box, the work–energy theorem
both ways, a force that rises then holds, a ramp's shape, a ball thrown up, a loop, a spring launcher, an engine's
force at top speed, and two carts that stick.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Work by several forces**: each is F d cos θ with its own angle to the motion (0° along, 180° against, 90°
across), and the net work is their sum. **The work–energy theorem**: net work = ½ m v² − ½ m u², read either way.
**Conservation**: ½ m v² = m g h, so the mass cancels. **Collisions**: momentum is kept; when the carts stick,
kinetic energy isn't.

In [ ]:
def work(F, d, theta_deg=0.0):
    return F * d * math.cos(math.radians(theta_deg))

def ke(m, v):
    return 0.5 * m * v * v

pull, fric, grav = work(100, 8), work(70, 8, 180), work(30 * g, 8, 90)          # P2.1
print(f"P2.1: pull {pull:.0f} J, friction {fric:.0f} J, gravity {grav:.0f} J (cos 90° = {math.cos(math.radians(90)):.0e}), "
      f"net {pull + fric + grav:.0f} J")
assert agrees("P2.1", [pull, fric, grav, pull + fric + grav])
assert agrees("P2.2", [(ke(1200, 25) - ke(1200, 10)) / 1000])                   # net work, in kJ
assert agrees("P2.3", [ke(80, 9) / 12])                                         # F d = ½ m v²
assert agrees("P2.4", [math.sqrt(2 * (0.5 * 2 * 30 + 30 * 3) / 4)])               # the area, then the speed
assert agrees("P2.5", [math.sqrt(2 * g * 3.2)])                                 # any ramp shape
assert agrees("P2.6", [12 ** 2 / (2 * g)])                                      # ½ m u² = m g h
assert agrees("P2.7", [math.sqrt(5 * g * 6)])                                   # √(5gR)
assert agrees("P2.8", [math.sqrt(2 * (0.5 * 500 * 0.04 ** 2) / 0.2)])           # ½ k x² = ½ m v²
assert agrees("P2.9", [45000 / 30])                                             # P = F v
v_stick = 3 * 4 / (3 + 1)                                                       # P2.10: momentum kept
assert agrees("P2.10", [v_stick, ke(3, 4) - ke(4, v_stick)])
print(f"P2.10: they move off at {v_stick:.2f} m/s; kinetic energy {ke(3, 4):.0f} J → {ke(4, v_stick):.0f} J")

### Two carts and a springy bumper: change the masses

A cart at 3 m/s meets a cart at rest through a stiff spring bumper (5,000 N/m). The simulator follows the squeeze
step by step. Watch three things: the total momentum (SKY) never moves; the kinetic energy (VIOLET) dips while the
bumper is squeezed, because the bumper holds some of it; and once they part, all of it is back. With 2 kg and 1 kg the
carts leave at 1 m/s and 4 m/s, and the kinetic energy dips to 6 J: the course's numbers. Make the masses equal and
the first cart stops dead; make the first one light and it bounces back.

In [ ]:
u1, k_b, gap = 3.0, 5000.0, 0.5

def bump_run(m1, m2, dt=1e-5):
    def deriv(t, s):
        x1, x2, v1, v2 = s
        c = max(0.0, gap - (x2 - x1)); F_ = k_b * c
        return (v1, v2, -F_ / m1, F_ / m2)
    return simulate(deriv, (0.0, 0.6, u1, 0.0), 0.3, dt)

def elastic(m1, m2, u):
    return (m1 - m2) / (m1 + m2) * u, 2 * m1 / (m1 + m2) * u

def collision_picture(m1=2.0, m2=1.0):
    run = bump_run(m1, m2, 1e-4)
    t = np.array([r[0] for r in run]); S = np.array([r[1] for r in run])
    c = np.clip(gap - (S[:, 1] - S[:, 0]), 0, None)
    K = 0.5 * m1 * S[:, 2] ** 2 + 0.5 * m2 * S[:, 3] ** 2; B = 0.5 * k_b * c ** 2
    p = m1 * S[:, 2] + m2 * S[:, 3]
    f1, f2 = elastic(m1, m2, u1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.8, bottom=0.15, wspace=0.25)
    a1.plot(t, S[:, 2], color=SKY, label=f"cart 1 ({m1:g} kg)"); a1.plot(t, S[:, 3], color=SKY, ls="--", label=f"cart 2 ({m2:g} kg)")
    a1.set_xlabel("time (s)"); a1.set_ylabel("velocity (m/s)"); a1.set_xlim(0, 0.3); a1.legend(frameon=False, loc="center right")
    a2.plot(t, K, color=VIOLET, label="kinetic energy"); a2.plot(t, B, color=VIOLET, ls=":", label="in the bumper")
    a2.plot(t, K + B, color=DIM, lw=1.4, label="the two together")
    i = int(np.argmin(K)); a2.scatter([t[i]], [K[i]], s=70, color=GOLD, zorder=4)
    a2.set_xlabel("time (s)"); a2.set_ylabel("energy (J)"); a2.set_xlim(0, 0.3); a2.set_ylim(0, 0.5 * m1 * u1 ** 2 * 1.15)
    a2.legend(frameon=False, loc="lower right")
    fig.suptitle(f"After: {S[-1, 2]:.2f} and {S[-1, 3]:.2f} m/s (the formula: {f1:.2f} and {f2:.2f}); momentum "
                 f"{p.min():.2f} to {p.max():.2f} kg m/s; kinetic energy dips to {K[i]:.2f} J", x=0.07, ha="left",
                 fontsize=14, color=GOLD)
    footnote(fig); plt.show()

_run = bump_run(2.0, 1.0)                                                       # the course's run, checked
_K = [0.5 * 2 * s[2] ** 2 + 0.5 * 1 * s[3] ** 2 for _, s in _run]
assert abs(_run[-1][1][2] - 1) < 1e-3 and abs(_run[-1][1][3] - 4) < 1e-3 and abs(min(_K) - 6) < 1e-3
assert max(abs(2 * s[2] + s[3] - 6) for _, s in _run) < 1e-9

In [ ]:
interact(collision_picture, m1=FloatSlider(value=2, min=0.5, max=5, step=0.5, description="m₁ (kg)"),
         m2=FloatSlider(value=1, min=0.5, max=5, step=0.5, description="m₂ (kg)"));

## Tier 3 · Stretch

Combine ideas, spot the trap, work backwards: an angle from the work it did, speed against mass, a force from the
speed it gave, heat on a slide, a toy loop's starting height, a rider's seat at the top of a loop, a buffer spring for
twice the speed, and an elastic collision with both checks.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards: `math.acos` gives an angle from its cosine, in radians, and `math.degrees` turns it back
into degrees. The rest is the same few formulas, rearranged. The last cell checks the
elastic collision both ways: momentum and kinetic energy before and after.

In [ ]:
theta = math.degrees(math.acos(707.11 / (50 * 20)))                             # P3.1: backwards to the angle
ratio = ke(1000, 20) / ke(2000, 10)                                             # P3.2: speed counts squared
F_start = 2 * ke(2, 6) / 3                                                      # P3.3: triangle area = ½ F0 L
heat = 40 * g * 3 - ke(40, 6)                                                   # P3.4: m g h − ½ m v²
h_start = 5 * g * 0.15 / (2 * g)                                                # P3.5: 2.5 R
v2_top = 16 ** 2 - 4 * g * 5                                                    # P3.6: speed² at the top
k_buf = [500 * v ** 2 / 0.3 ** 2 for v in (3, 6)]                               # P3.7: k = m v²/x²
v1, v2 = elastic(1, 4, 5)                                                       # P3.8
print(f"θ = {theta:.2f}° · A has {ratio:.0f}× the energy · F0 = {F_start:.0f} N · heat {heat:.0f} J, friction "
      f"{heat / 6:.0f} N · h = {h_start:.3f} m · top {math.sqrt(v2_top):.2f} m/s, seat {v2_top / (g * 5) - 1:.2f} × weight")
print(f"P3.8: {v1:.0f} and {v2:.0f} m/s; momentum {1 * 5} = {1 * v1 + 4 * v2:.0f}; kinetic energy {ke(1, 5)} = "
      f"{ke(1, v1) + ke(4, v2)}")
for k, v in (("P3.1", [theta]), ("P3.2", [ratio]), ("P3.3", [F_start]), ("P3.4", [heat, heat / 6]),
             ("P3.5", [h_start]), ("P3.6", [math.sqrt(v2_top), v2_top / (g * 5) - 1]), ("P3.7", k_buf), ("P3.8", [v1, v2])):
    assert agrees(k, v), k
assert 1 * 5 == 1 * v1 + 4 * v2 and ke(1, 5) == ke(1, v1) + ke(4, v2)

## The roller coaster's loop: change the launch speed and the radius

The course opened with a question: how fast must a roller coaster launch to carry its riders round a loop twenty
metres tall, pressed into their seats? Its launch track runs level into the bottom of the loop, and friction is
ignored. At the top, v²/R must be at least g, so at the bottom v² must be at least 5gR: for a 10 m radius, √490 =
22.14 m/s. At exactly that speed the seat's push at the top is zero, and the riders float over it.

The simulator runs the train round the loop, step by step, with gravity alone (the track's push is always across the
path, so it does no work). The energy bars show, per kilogram of the train, where the launch's kinetic energy is at the
point you choose: still moving (K) or lifted (U). The faint line is the launch's energy: the two bars always add up to
it. The title gives the speed at the top and how hard the seat presses there, as a share of the riders' weight. At
24 m/s and 10 m it's 13.56 m/s and 0.88. Try 21 m/s: the seat's push drops below zero, and the riders would hang in
their lap bars. Below about 19.8 m/s the train doesn't reach the top at all.

In [ ]:
# @COASTER
COASTER = {'R': 10.0, 'v': 24.0}
# @END COASTER

def loop_run(V, R, dt=1e-3):
    """The train round the loop (θ from the loop's bottom), launched at V on the level; it stops at the end of the loop,
    or where it would stall."""
    deriv = lambda t, s: (s[1], -g / R * math.sin(s[0]))
    return simulate(deriv, (0.0, V / R), 30, dt, stop=lambda t, s: s[0] >= 2 * math.pi or s[1] <= 0)

def state_at(run, theta):
    """θ̇ where the run passes θ (None if it never gets there)."""
    for (t0, s0), (t1, s1) in zip(run, run[1:]):
        if s0[0] <= theta <= s1[0] and s1[0] > s0[0]:
            f = (theta - s0[0]) / (s1[0] - s0[0]); return s0[1] + f * (s1[1] - s0[1])
    return None

def loop_picture(V=24.0, R=10.0, where=180.0):
    run = loop_run(V, R)
    th = np.array([s[0] for _, s in run])
    top = state_at(run, math.pi)
    w = state_at(run, math.radians(where))
    K0 = 0.5 * V * V                                                     # the launch's kinetic energy, per kilogram
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.25, 1]})
    fig.subplots_adjust(left=0.05, right=0.98, top=0.83, bottom=0.12, wspace=0.2)
    a1.set_aspect("equal"); a1.grid(False); a1.set_xticks([]); a1.set_yticks([])
    for side in ("left", "bottom"): a1.spines[side].set_visible(False)
    full = np.linspace(0, 2 * np.pi, 200)
    a1.plot(R * np.sin(full), R * (1 - np.cos(full)), color=GHOST, lw=6)
    a1.plot([-2.2 * R, 2.2 * R], [0, 0], color=GHOST, lw=6)
    a1.plot(R * np.sin(th), R * (1 - np.cos(th)), color=CHALK, lw=3)
    if w is not None:
        x_, y_ = R * math.sin(math.radians(where)), R * (1 - math.cos(math.radians(where)))
        a1.scatter([x_], [y_], s=160, color=GOLD, zorder=4)
        a1.annotate(f"{R * w:.1f} m/s", (x_, y_), textcoords="offset points", xytext=(12, -4), color=GOLD, fontsize=12)
    a1.set_xlim(-2.3 * R, 2.3 * R); a1.set_ylim(-0.3 * R, 2.3 * R)
    if w is None:                                                        # it stalls first: the bars where it stops
        th_w, w = run[-1][1]; label = f"it stalls at {math.degrees(th_w):.0f}°, before {where:.0f}°"
    else:
        th_w = math.radians(where); label = f"{where:.0f}° round the loop"
    K, U = 0.5 * (R * w) ** 2, g * R * (1 - math.cos(th_w))
    a2.bar(["K", "U"], [K, U], color=VIOLET)
    a2.axhline(K0, color=DIM, ls="--", lw=1.4)
    a2.set_ylabel("energy per kilogram (J/kg)"); a2.set_ylim(0, max(K0 * 1.15, 1.0))
    a2.set_title(label, fontsize=13, color=CHALK)
    if top is None:
        title = f"Launch {V:g} m/s, loop R = {R:g} m: the train stalls before the top"
    else:
        seat = (R * top) ** 2 / (g * R) - 1
        title = (f"Launch {V:g} m/s, loop R = {R:g} m: top speed {R * top:.2f} m/s, the seat presses with "
                 f"{seat:.2f} × the weight" + ("" if seat >= 0 else " (the riders would hang in their lap bars)"))
    fig.suptitle(title, x=0.05, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

# the course's run, checked: the top speed against v² − 4gR, and K + U always the launch's energy
_run = loop_run(COASTER["v"], COASTER["R"], 1e-4)
_top = COASTER["R"] * state_at(_run, math.pi)
assert abs(_top ** 2 - (COASTER["v"] ** 2 - 4 * g * COASTER["R"])) < 1e-4
assert abs(_top - 13.564660) < 1e-5 and round(_top ** 2 / (g * COASTER["R"]) - 1, 2) == 0.88
assert round(math.sqrt(5 * g * COASTER["R"]), 2) == 22.14                     # the least launch speed
for _, (_th, _w) in _run[::500]:
    _E = 0.5 * (COASTER["R"] * _w) ** 2 + g * COASTER["R"] * (1 - math.cos(_th))
    assert abs(_E - 0.5 * COASTER["v"] ** 2) < 1e-6 * 288
print(f"launched at {COASTER['v']:g} m/s: {_top:.2f} m/s at the top; the least launch is "
      f"{math.sqrt(5 * g * COASTER['R']):.2f} m/s")

In [ ]:
interact(loop_picture, V=FloatSlider(value=24, min=18, max=30, step=0.5, description="launch (m/s)"),
         R=FloatSlider(value=10, min=5, max=16, step=0.5, description="radius (m)"),
         where=FloatSlider(value=180, min=0, max=360, step=5, description="where (°)"));

## Sources
Every exercise and set-up here is our own. The chapter order
follows the standard school course on work, energy and power; the definitions agree with OpenStax, University Physics
volume 1, chapter 7 "Work and Kinetic Energy", chapter 8 "Potential Energy and Conservation of Energy" and chapter 9
"Linear Momentum and Collisions" (retrieved 2026-11-11). Every simulation is the course's fourth-order Runge–Kutta
simulator, checked against the formula it illustrates.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/ph05_work_energy/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("ph05_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"]) == (_e["q"], _e["working"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
        assert all(abs(a - float(b_)) < 1e-9 for a, b_ in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX) and _P.COASTER == COASTER and _P.g == g
    assert abs(_top - _P.COASTER_SIM_TOP) < 1e-6 and abs(math.sqrt(5 * g * COASTER["R"]) - _P.COASTER_VMIN) < 1e-12
    print("matches the course's practice set")